In [1]:
import geopandas as gpd
import numpy as np
import pandas as pd
import requests 
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import plotly.express as px

In [2]:
# Tabla de migras
df_migras_90_24 = pd.read_csv("migras_90_24.csv")
df_migras_90_24.head()

,cod_orig,iso2_orig,iso3_orig,origen_ES,origen_EN,region_orig_ES,region_orig_EN,subregion_orig_ES,subregion_orig_EN,poblacion_orig,...,region_des_EN,subregion_des_ES,subregion_des_EN,poblacion_des,menos_desarr_des,sin_litoral_des,lon_des,lat_des,año,migrantes
0,20,AD,AND,Andorra,Andorra,Europa,Europe,Europa meridional,Southern Europe,52000.0,...,Europe,Europa septentrional,Northern Europe,4986000.0,False,False,23.313225,61.982007,1990,1
1,20,AD,AND,Andorra,Andorra,Europa,Europe,Europa meridional,Southern Europe,52000.0,...,Europe,Europa septentrional,Northern Europe,4241000.0,False,False,11.478464,61.439017,1990,2
2,20,AD,AND,Andorra,Andorra,Europa,Europe,Europa meridional,Southern Europe,52000.0,...,Europe,Europa meridional,Southern Europe,10258000.0,False,False,22.597281,39.514859,1990,6
3,20,AD,AND,Andorra,Andorra,Europa,Europe,Europa meridional,Southern Europe,52000.0,...,Europe,Europa meridional,Southern Europe,9989000.0,False,False,-7.956154,39.725761,1990,122
4,20,AD,AND,Andorra,Andorra,Europa,Europe,Europa meridional,Southern Europe,52000.0,...,Americas,América Latina y el Caribe,Latin America and the Caribbean,7130000.0,False,True,-64.659013,-16.777415,1990,2


In [3]:
nombres_ES = pd.read_csv("nombres_ES.csv")
nombres_ES.head()

,cod_m49,iso3,name_en,name_es,subregion,region
0,100,BGR,Bulgaria,Bulgaria,Europa Oriental,Europa
1,104,MMR,Myanmar,Myanmar,Asia Sudoriental,Asia
2,108,BDI,Burundi,Burundi,África Oriental,África
3,112,BLR,Belarus,Belarús,Europa Oriental,Europa
4,116,KHM,Cambodia,Camboya,Asia Sudoriental,Asia


In [4]:
# df_nuevo es el renombrado con nombres más cortos
df_nuevo = df_migras_90_24.copy()

mapa_paises = nombres_ES.set_index('iso3')['name_es']

df_nuevo['origen_ES'] = df_nuevo['iso3_orig'].map(mapa_paises)
df_nuevo['destino_ES'] = df_nuevo['iso3_des'].map(mapa_paises)

df_nuevo = df_nuevo[df_nuevo["origen_ES"] != "Otros"]
# Basicamente utilicé los otros nombres y saqué el otros del análisis

df_nuevo

,cod_orig,iso2_orig,iso3_orig,origen_ES,origen_EN,region_orig_ES,region_orig_EN,subregion_orig_ES,subregion_orig_EN,poblacion_orig,...,region_des_EN,subregion_des_ES,subregion_des_EN,poblacion_des,menos_desarr_des,sin_litoral_des,lon_des,lat_des,año,migrantes
0,20,AD,AND,Andorra,Andorra,Europa,Europe,Europa meridional,Southern Europe,52000.0,...,Europe,Europa septentrional,Northern Europe,4986000.0,False,False,23.313225,61.982007,1990,1
1,20,AD,AND,Andorra,Andorra,Europa,Europe,Europa meridional,Southern Europe,52000.0,...,Europe,Europa septentrional,Northern Europe,4241000.0,False,False,11.478464,61.439017,1990,2
2,20,AD,AND,Andorra,Andorra,Europa,Europe,Europa meridional,Southern Europe,52000.0,...,Europe,Europa meridional,Southern Europe,10258000.0,False,False,22.597281,39.514859,1990,6
3,20,AD,AND,Andorra,Andorra,Europa,Europe,Europa meridional,Southern Europe,52000.0,...,Europe,Europa meridional,Southern Europe,9989000.0,False,False,-7.956154,39.725761,1990,122
4,20,AD,AND,Andorra,Andorra,Europa,Europe,Europa meridional,Southern Europe,52000.0,...,Americas,América Latina y el Caribe,Latin America and the Caribbean,7130000.0,False,True,-64.659013,-16.777415,1990,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63792,716,ZW,ZWE,Zimbabwe,Zimbabwe,África,Africa,África Subsahariana,Sub-Saharan Africa,16634000.0,...,Americas,América Latina y el Caribe,Latin America and the Caribbean,12413000.0,False,True,-64.659013,-16.777415,2024,3
63793,716,ZW,ZWE,Zimbabwe,Zimbabwe,África,Africa,África Subsahariana,Sub-Saharan Africa,16634000.0,...,Americas,América Latina y el Caribe,Latin America and the Caribbean,28405000.0,False,False,-66.165784,7.141976,2024,12
63794,716,ZW,ZWE,Zimbabwe,Zimbabwe,África,Africa,África Subsahariana,Sub-Saharan Africa,16634000.0,...,Americas,América septentrional,Northern America,39742000.0,False,False,-104.215816,59.833782,2024,14291
63795,716,ZW,ZWE,Zimbabwe,Zimbabwe,África,Africa,África Subsahariana,Sub-Saharan Africa,16634000.0,...,Oceania,Australia y Nueva Zelandia,Australia and New Zealand,26713000.0,False,False,134.437438,-26.026147,2024,47195


# **1. Creación de archivos de Gephi**

In [5]:
# Colocamos unicamente territorios que reciban y emitan migrantes, por lo tanto eliminamos "Otros". 
df_gephi = df_nuevo[df_nuevo["origen_ES"] != "Otros"].copy()

df_gephi = df_gephi[[
    "region_orig_ES",
    "region_des_ES",
    "iso3_orig",
    "iso3_des",
    "origen_ES",
    "destino_ES",
    "año",
    "migrantes",
    "poblacion_orig",
    "poblacion_des",
]].copy()

df_gephi

,region_orig_ES,region_des_ES,iso3_orig,iso3_des,origen_ES,destino_ES,año,migrantes,poblacion_orig,poblacion_des
0,Europa,Europa,AND,FIN,Andorra,Finlandia,1990,1,52000.0,4986000.0
1,Europa,Europa,AND,NOR,Andorra,Noruega,1990,2,52000.0,4241000.0
2,Europa,Europa,AND,GRC,Andorra,Grecia,1990,6,52000.0,10258000.0
3,Europa,Europa,AND,PRT,Andorra,Portugal,1990,122,52000.0,9989000.0
4,Europa,Américas,AND,BOL,Andorra,Bolivia,1990,2,52000.0,7130000.0
...,...,...,...,...,...,...,...,...,...,...
63792,África,Américas,ZWE,BOL,Zimbabwe,Bolivia,2024,3,16634000.0,12413000.0
63793,África,Américas,ZWE,VEN,Zimbabwe,Venezuela,2024,12,16634000.0,28405000.0
63794,África,Américas,ZWE,CAN,Zimbabwe,Canadá,2024,14291,16634000.0,39742000.0
63795,África,Oceanía,ZWE,AUS,Zimbabwe,Australia,2024,47195,16634000.0,26713000.0


### a) Construcción de Grafos globales 1990 y 2024

In [6]:
def preparar_gephi(df, año):
    df_year = df[df["año"] == año].copy()
    
    # Definimos las aristas
    aristas = (
        df_year
        .groupby(["iso3_orig", "iso3_des"], as_index=False)
        .agg({"migrantes": "sum"})
        .rename(columns={
            "iso3_orig": "Source",
            "iso3_des": "Target",
            "migrantes": "Weight"
        })
    )
    
    # Definimos los nodos
    nodos_orig = df_year[["iso3_orig", "origen_ES", "poblacion_orig"]]\
        .rename(columns={
            "iso3_orig": "Id",
            "origen_ES": "Label",
            "poblacion_orig": "Population"
        })

    nodos_des = df_year[["iso3_des", "destino_ES", "poblacion_des"]]\
        .rename(columns={
            "iso3_des": "Id",
            "destino_ES": "Label",
            "poblacion_des": "Population"
        })

    nodos = (
        pd.concat([nodos_orig, nodos_des])
        .drop_duplicates(subset="Id")
    )
    
    return nodos, aristas

# Para 1990
nodes_1990, edges_1990 = preparar_gephi(df_gephi, 1990)

# Luego, para 2024
nodes_2024, edges_2024 = preparar_gephi(df_gephi, 2024)

### Filtrado de aristas con mayor peso

In [7]:
# Se aproximan 82 nodos por año lo que equivale a migraciones con un peso mayor a 281000 y 420000 respecto a 1990 y 2024. 
edges_1990_filtrado = edges_1990[edges_1990["Weight"] > 281000]
nodos_usados_1990 = pd.unique(edges_1990_filtrado[["Source", "Target"]].values.ravel())
nodes_1990_filtrado = nodes_1990[nodes_1990["Id"].isin(nodos_usados_1990)]

edges_2024_filtrado = edges_2024[edges_2024["Weight"] > 420000]
nodos_usados_2024 = pd.unique(edges_2024_filtrado[["Source", "Target"]].values.ravel())
nodes_2024_filtrado = nodes_2024[nodes_2024["Id"].isin(nodos_usados_2024)]

In [8]:
# Exportar nodos y aristas según año
#nodes_1990_filtrado.to_csv("nodos_global_1990.csv", index=False)
#edges_1990_filtrado.to_csv("aristas_global_1990.csv", index=False)

#nodes_2024_filtrado.to_csv("nodos_global_2024.csv", index=False)
#edges_2024_filtrado.to_csv("aristas_global_2024.csv", index=False)

### b) Analisis Intracontinental de 2024.

In [9]:
# Colocamos unicamente el año a ser analizado
df_gephi_2024 = df_gephi[df_gephi["año"]==2024]
df_gephi_2024

df_intra_2024 = df_gephi_2024[
    (df_gephi_2024['region_orig_ES'] == df_gephi_2024['region_des_ES'])
].copy()

tablas_de_continente = {}

for continente, datos in df_intra_2024.groupby('region_orig_ES'):
    top3_cont = (
        datos.sort_values(['origen_ES', 'migrantes'], ascending=[True, False])
        .groupby('origen_ES')
        .head(3) 
        #El criterio consiste en seleccionar, para cada país de origen dentro de cada continente, los tres destinos con mayor cantidad de migrantes
    )
    
    # Guardamos la tabla limpia en el diccionario
    tablas_de_continente[continente] = top3_cont.reset_index(drop=True)

# Acá cambiamos unicamente la variable "continente" y así obtenemos sus nodos y aristas correspondientes
continente = "África" # Se selecciona entre: Américas | Asia | Oceanía | África | Europa 
df = tablas_de_continente[continente]

nodos = pd.concat([
    df[['origen_ES']].rename(columns={'origen_ES': 'Id'}),
    df[['destino_ES']].rename(columns={'destino_ES': 'Id'})
]).drop_duplicates().reset_index(drop=True)

nodos["Label"] = nodos["Id"] # Agregamos Label igual a Id, facilita la identificación en Gephi

aristas = df.rename(columns={
    'origen_ES':'Source',
    'destino_ES':'Target',
    'migrantes':'Weight'
})[['Source','Target','Weight']]

# Pasar a CSV para importar en Gephi
# nodos.to_csv(f'nodos{continente}.csv', index=False)
# aristas.to_csv(f'aristas{continente}.csv', index=False)

# **2-. Migración Limítrofe**

In [10]:
import geopandas as gpd

# De acá cargué los datos:
url = "https://naturalearth.s3.amazonaws.com/50m_cultural/ne_50m_admin_0_countries.zip"
mundial = gpd.read_file(url)

mundial = mundial[['NAME', 'ISO_A3_EH', 'CONTINENT', 'geometry']].copy()
mundial = mundial.rename(columns={'ISO_A3_EH': 'ISO'})

print(mundial.head())

        NAME  ISO      CONTINENT  \
0   Zimbabwe  ZWE         Africa   
1     Zambia  ZMB         Africa   
2      Yemen  YEM           Asia   
3    Vietnam  VNM           Asia   
4  Venezuela  VEN  South America   

                                            geometry  
0  POLYGON ((31.28789 -22.40205, 31.19727 -22.344...  
1  POLYGON ((30.39609 -15.64307, 30.25068 -15.643...  
2  MULTIPOLYGON (((53.08564 16.64839, 52.58145 16...  
3  MULTIPOLYGON (((104.06396 10.39082, 104.08301 ...  
4  MULTIPOLYGON (((-60.82119 9.13838, -60.94141 9...  


### **Unificación de datos**
Basicamente me quedaba Guayana Francesa como parte de Francia asi que lo separé y ahí me quedó la columna con si es limítrofe o no

In [11]:
# Utilizo solo países en el datataset df_nuevo
codigos_migracion = df_nuevo['iso3_orig'].unique()
mundial_filtrado = mundial[mundial['ISO'].isin(codigos_migracion)].copy()

# Vemos cuales países no han sido considerados en el dataset
print("Cantidad de países en mundial_filtrado: ", mundial_filtrado["ISO"].nunique())
print("Cantidad de países en df_nuevo: ", df_nuevo["iso3_orig"].nunique())

iso_orig = set(df_nuevo["iso3_orig"].unique())
iso_mundial = set(mundial_filtrado["ISO"].unique())

faltantes_en_mundial = iso_orig - iso_mundial

print("Códigos en df_nuevo pero no en mundial:")
print(faltantes_en_mundial)

faltantes_en_df = iso_mundial-iso_orig
print("Códigos en mundial pero no en df_nuevo:")
print(faltantes_en_df)

Cantidad de países en mundial_filtrado:  200
Cantidad de países en df_nuevo:  201
Códigos en df_nuevo pero no en mundial:
{'GUF'}
Códigos en mundial pero no en df_nuevo:
set()


In [12]:
# Creamos una lista con el pais y sus países limítrofes, y separamos Guayana Francesa ya que está integrada dentro de Francia
lista_relaciones = []

for idx, fila in mundial_filtrado.iterrows():
    iso_actual = fila['ISO']
    geometria_actual = fila['geometry']
    continente_actual = fila['CONTINENT']   # Necesario para poder catologar el caso de Guayana Francesa
    
    vecinos = mundial_filtrado[mundial_filtrado.geometry.touches(geometria_actual) & (mundial_filtrado['ISO'] != iso_actual)]
    codigos_vecinos = vecinos['ISO'].tolist()

    # Caso particular: si el país pertenece a América del Sur y su vecino es Francia, entonces estamos hablando de Guayana Francesa.
    if continente_actual == 'South America': codigos_vecinos = ['GUF' if x == 'FRA' else x for x in codigos_vecinos]
    
    # Guardamos en nuestra lista
    lista_relaciones.append({'Pais': iso_actual, 'Vecinos': codigos_vecinos,'Cantidad': len(codigos_vecinos)})
    
# Agregamos la región faltante
lista_relaciones.append({'Pais': 'GUF', 'Vecinos': ['BRA', 'SUR'], 'Cantidad': 2})

# lista_relaciones <- Queda en formato lista, por si hace falta verificar

In [13]:
# Luego, lo pasamos a diccionario para acceder más rápido y creamos la columna "Es_limitrofe"
vecinos_dict = {item['Pais']: item['Vecinos'] for item in lista_relaciones}

df_nuevo.loc[:, 'Es_Limitrofe'] = df_nuevo.copy().apply(
    lambda fila: fila['iso3_des'] in vecinos_dict.get(fila['iso3_orig'], []),
    axis=1
)
display(df_nuevo.head())

,cod_orig,iso2_orig,iso3_orig,origen_ES,origen_EN,region_orig_ES,region_orig_EN,subregion_orig_ES,subregion_orig_EN,poblacion_orig,...,subregion_des_ES,subregion_des_EN,poblacion_des,menos_desarr_des,sin_litoral_des,lon_des,lat_des,año,migrantes,Es_Limitrofe
0,20,AD,AND,Andorra,Andorra,Europa,Europe,Europa meridional,Southern Europe,52000.0,...,Europa septentrional,Northern Europe,4986000.0,False,False,23.313225,61.982007,1990,1,False
1,20,AD,AND,Andorra,Andorra,Europa,Europe,Europa meridional,Southern Europe,52000.0,...,Europa septentrional,Northern Europe,4241000.0,False,False,11.478464,61.439017,1990,2,False
2,20,AD,AND,Andorra,Andorra,Europa,Europe,Europa meridional,Southern Europe,52000.0,...,Europa meridional,Southern Europe,10258000.0,False,False,22.597281,39.514859,1990,6,False
3,20,AD,AND,Andorra,Andorra,Europa,Europe,Europa meridional,Southern Europe,52000.0,...,Europa meridional,Southern Europe,9989000.0,False,False,-7.956154,39.725761,1990,122,False
4,20,AD,AND,Andorra,Andorra,Europa,Europe,Europa meridional,Southern Europe,52000.0,...,América Latina y el Caribe,Latin America and the Caribbean,7130000.0,False,True,-64.659013,-16.777415,1990,2,False


---
### **Resultados finales**

In [14]:
tabla_limitrofes = df_nuevo[df_nuevo['Es_Limitrofe']].pivot_table(
    index='region_orig_ES',
    columns='año',
    values='migrantes',
    aggfunc='sum'
).fillna(0)

tabla_totales = df_nuevo.pivot_table( # Estas tablas sirven para verificar los datos, tanto de limítrofes como totales
    index='region_orig_ES',
    columns='año',
    values='migrantes',
    aggfunc='sum'
).fillna(0)

for tabla in [tabla_limitrofes, tabla_totales]:
    tabla.drop('TOTAL MUNDIAL', errors='ignore', inplace=True)
    tabla.loc['TOTAL MUNDIAL'] = tabla.sum(axis=0)


def formatear_combinado(valor_lim, valor_total): # Útil para representar los números de forma más sencilla. Por ejemplo acá se ve con M
    if valor_total == 0 or pd.isna(valor_total):
        return "0 (0%)"
    
    porcentaje = (valor_lim / valor_total) * 100
    
    # números grandes a Millones (M) o miles (k)
    if valor_lim >= 1_000_000:
        val_str = f"{valor_lim/1_000_000:.1f}M"
    elif valor_lim >= 1_000:
        val_str = f"{valor_lim/1_000:.1f}k"
    else:
        val_str = f"{int(valor_lim)}"
        
    return f"{val_str} ({porcentaje:.1f}%)"
    
# Crear tabla visual (valor + porcentaje)
tabla_visual = tabla_limitrofes.copy()
anios = tabla_limitrofes.columns

for anio in anios:
    tabla_visual[anio] = tabla_limitrofes.apply(
        lambda row: formatear_combinado(row[anio], tabla_totales.loc[row.name, anio]),
        axis=1
    )

# % Promedio
tabla_visual['% Promedio'] = (
    (tabla_limitrofes[anios].mean(axis=1) /
     tabla_totales[anios].mean(axis=1)) * 100
).round(1)

# Variación %. Variacion  (valor anterior - valor nuevo / valor anterior) 
primer_anio, ultimo_anio = anios.min(), anios.max()

tabla_visual['Var_Total_%'] = (
    (tabla_limitrofes[ultimo_anio] - tabla_limitrofes[primer_anio]) /
    tabla_limitrofes[primer_anio].replace(0, np.nan) * 100
).fillna(0).round(1)

# Correlación
def calcular_corr_segura(df):

    res = df.groupby('año').agg(
        migrantes_total=('migrantes', 'sum'),
        migrantes_lim=('migrantes', lambda x: x[df.loc[x.index,'Es_Limitrofe']].sum())
    )

    res['prop_lim'] = res['migrantes_lim'] / res['migrantes_total'].replace(0, np.nan)
    res = res.dropna(subset=['prop_lim','migrantes_total'])

    if len(res) < 2 or res['prop_lim'].std() == 0 or res['migrantes_total'].std() == 0:
        return np.nan

    return res['prop_lim'].corr(res['migrantes_total'])

correlaciones = {}

for region in df_nuevo['region_orig_ES'].unique():
    val = calcular_corr_segura(df_nuevo[df_nuevo['region_orig_ES']==region])
    correlaciones[region] = round(val,3) if pd.notnull(val) else "N/A"

val_global = calcular_corr_segura(df_nuevo)
correlaciones['TOTAL MUNDIAL'] = round(val_global,3) if pd.notnull(val_global) else "N/A"

tabla_visual['Corr_Lim_Total'] = tabla_visual.index.map(correlaciones)

tabla_visual

año,1990,1995,2000,2005,2010,2015,2020,2024,% Promedio,Var_Total_%,Corr_Lim_Total
region_orig_ES,,,,,,,,,,,
Américas,8.7M (51.0%),11.4M (52.2%),13.6M (51.7%),16.0M (50.4%),17.9M (48.7%),18.4M (47.1%),19.8M (42.9%),21.6M (41.1%),46.9,148.0,-0.932
Asia,27.3M (50.4%),25.0M (44.5%),26.0M (41.4%),26.0M (37.2%),29.8M (35.0%),33.0M (32.6%),35.7M (32.2%),37.6M (31.2%),36.4,37.7,-0.901
Europa,24.7M (50.8%),23.5M (48.3%),22.5M (48.3%),21.5M (45.2%),20.8M (41.5%),19.7M (38.4%),19.0M (35.8%),21.4M (35.4%),42.6,-13.2,-0.797
África,11.5M (58.5%),11.9M (56.2%),10.3M (48.9%),10.4M (43.3%),11.4M (41.0%),14.7M (42.6%),17.4M (43.8%),20.9M (45.8%),46.4,81.5,-0.560
TOTAL MUNDIAL,72.2M (51.5%),71.8M (48.3%),72.5M (45.9%),73.9M (42.4%),79.9M (39.8%),85.7M (37.7%),92.0M (36.6%),101.5M (36.2%),41.1,40.6,-0.932


---

### **Apartado de prueba**

In [15]:
# 1. Agrupar todos los datos por año (sin importar la región)
resumen_mundial = df_nuevo.groupby('año').agg(
    migrantes_total=('migrantes', 'sum'),
    migrantes_lim=('migrantes', lambda x: x[df_nuevo.loc[x.index, 'Es_Limitrofe']].sum())
)

# 2. Calcular la proporción global
resumen_mundial['prop_lim'] = resumen_mundial['migrantes_lim'] / resumen_mundial['migrantes_total']

# 3. Calcular la correlación de Pearson
corr_mundial = resumen_mundial['prop_lim'].corr(resumen_mundial['migrantes_total'])

print(f"--- Análisis Mundial ---")
print(f"Correlación (Volumen Total vs % Limítrofe): {corr_mundial:.3f}")
print("\nDatos por año:")
print(resumen_mundial[['migrantes_total', 'prop_lim']])

--- Análisis Mundial ---
Correlación (Volumen Total vs % Limítrofe): -0.932

Datos por año:
      migrantes_total  prop_lim
año                            
1990        140170551  0.515030
1995        148791873  0.482546
2000        157899241  0.458892
2005        174207385  0.424038
2010        201011349  0.397555
2015        227354550  0.377136
2020        251602578  0.365645
2024        280717024  0.361531


In [16]:
# Acá para ver los corredores que tienen mas cantidad limítrofe por continente y año
def top_limitrofes(df, region, anio, n=10):
    
    df_filtrado = df[
        (df['año'] == anio) &
        (df['region_orig_ES'] == region) &
        (df['Es_Limitrofe'] == True)
    ]
    
    top = (
        df_filtrado
        .groupby(['origen_ES','destino_ES'])['migrantes']
        .sum()
        .sort_values(ascending=False)
        .head(n)
        .reset_index()
    )
    
    return top
top_limitrofes(df_nuevo, "Asia", 1990) # Por ejemplo Asia, Bangladesh -> India, Afganistan -> Iran y Pakistan

,origen_ES,destino_ES,migrantes
0,Bangladesh,India,4061412
1,Afganistán,Irán,3980307
2,Afganistán,Pakistán,3374973
3,India,Pakistán,2818248
4,Kazajstán,Rusia,2349697
5,Pakistán,India,1826106
6,China,Hong Kong,1659157
7,Azerbaiyán,Rusia,936852
8,Palestina,Jordania,929992
9,Georgia,Rusia,656888


In [17]:
tabla_totales

año,1990,1995,2000,2005,2010,2015,2020,2024
region_orig_ES,,,,,,,,
Américas,17059704,21776759,26259049,31848183,36733954,38973664,46175381,52402073
Asia,54180953,56281639,62896712,69801805,85226114,101116388,111068997,120718957
Europa,48532106,48688056,46668943,47459898,50056975,51312797,53031571,60502939
Oceanía,746718,850400,982620,1106254,1264543,1399730,1454406,1505219
África,19651070,21195019,21091917,23991245,27729763,34551971,39872223,45587836
TOTAL MUNDIAL,140170551,148791873,157899241,174207385,201011349,227354550,251602578,280717024


In [18]:
tabla_limitrofes

año,1990,1995,2000,2005,2010,2015,2020,2024
region_orig_ES,,,,,,,,
Américas,8692448,11375619,13586112,16046216,17899013,18358417,19825205,21557149
Asia,27332387,25017302,26011346,25956070,29844723,32954494,35724723,37634239
Europa,24667588,23497808,22548211,21469732,20787699,19722718,18999419,21421142
África,11499616,11908181,10312994,10398513,11381583,14708010,17447778,20875355
TOTAL MUNDIAL,72192039,71798910,72458663,73870531,79913018,85743639,91997125,101487885
